In [ ]:
import pandas as pd
import os

import pickle
from pathlib import Path

In [ ]:
#GET SAMPLE PCs

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

In [ ]:
def wrangle_pcs(pc_df): 
    
    ancestry_pred = pd.read_csv("ancestry_preds.tsv", delimiter="\t")
    
    pc_df["pca_features"] = pc_df["pca_features"].str[1:-1]
    
    PCs = pc_df["pca_features"].str.split(",", n = 16, expand = True)
    PCs = PCs.astype(float)
    
    pid = pc_df[["research_id"]]
    
    columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
    PCs.columns = columns
    
    PCs_final = pd.concat([pid, PCs], axis = 1)
    
    PCs_final.to_csv('wrangled_ancestry_pcs.csv')
    
    return PCs_final
    
    

In [ ]:
wrangled_pcs = wrangle_pcs(ancestry_pred)
wrangled_pcs

In [ ]:
#save file
wrangled_pcs.to_csv('wrangled_ancestry_pcs.csv')

In [ ]:
#With PCs

import pandas as pd
import numpy as np
import re

# ======= EDIT THESE to match your data =======
ID_CASES     = "person_id"     # ID in cases_df
ID_CONTROLS  = "person_id"     # ID in controls_df
SEX_CASES    = "sex_at_birth"  # sex in cases_df (M/F or 1/2)
SEX_CONTROLS = "sex_at_birth"  # sex in controls_df (M/F or 1/2)

# PCs table (already loaded as PCs_final)
PC_ID_COL    = "research_id"   # ID column in PCs_final
PC_PREFIX    = "PC"            # PC columns start with something like PC1, pc_02, "PC 3", etc.
# ============================================

def normalize_pc_columns(pcs_df, pc_id_col, prefix="PC"):
    """Rename variants of PC columns to 'PC<k>' (pc1, PC_01, 'PC 2' -> PC1/PC2), keep only ID + PCs."""
    df = pcs_df.copy()
    # strip spaces around names
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        if c == pc_id_col:
            continue
        s = str(c).strip()
        # match pc, PC, PC_, PC<space> and zero-padded numbers
        m = re.match(r'(?i)^pc[\s_]*0*([0-9]+)$', s)
        if not m:
            # also handle 'principal component 1' / 'pcscore1' styles if present
            m = re.match(r'(?i)^(principal[\s_]*component|pcscore)[\s_]*0*([0-9]+)$', s)
            if m:
                num = m.group(2)
                rename[c] = f'PC{int(num)}'
                continue
        if m:
            num = m.group(1)
            rename[c] = f'PC{int(num)}'
    df = df.rename(columns=rename)

    # keep ID + PC* columns only
    pc_cols = [c for c in df.columns if c != pc_id_col and str(c).upper().startswith(prefix.upper())]
    # ensure numeric PCs
    for c in pc_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # sort PC columns in numeric order (PC1, PC2, ..., PC10)
    def pc_num(name):
        m = re.search(r'(\d+)$', name)
        return int(m.group(1)) if m else 10**9

    pc_cols = sorted(pc_cols, key=pc_num)
    return df[[pc_id_col] + pc_cols], pc_cols

# ---------- 1) Force ID dtypes to string everywhere ----------
cases = cases.copy()
ctrls = ctrls.copy()
PCs_final = PCs_final.copy()

cases[ID_CASES] = cases[ID_CASES].astype("string")
ctrls[ID_CONTROLS] = ctrls[ID_CONTROLS].astype("string")
PCs_final[PC_ID_COL] = PCs_final[PC_ID_COL].astype("string")

# ---------- 2) Base sample list (union of IDs; FID=IID) ----------
case_ids = pd.Series(cases[ID_CASES].dropna().unique(), dtype="string", name="IID")
ctrl_ids = pd.Series(ctrls[ID_CONTROLS].dropna().unique(), dtype="string", name="IID")
base = pd.DataFrame({"IID": pd.concat([case_ids, ctrl_ids], ignore_index=True).drop_duplicates()})
base["FID"] = base["IID"]
base = base.astype({"FID":"string","IID":"string"})[["FID","IID"]]

# ---------- 3) Phenotype (1=case, 0=control, NA otherwise) ----------
pheno = base.copy()
case_set, ctrl_set = set(case_ids), set(ctrl_ids)
pheno["meningitis"] = np.where(
    pheno["IID"].isin(case_set), 1,
    np.where(pheno["IID"].isin(ctrl_set), 0, np.nan)
)
pheno.to_csv("pheno.tsv", sep="\t", index=False)

# ---------- 4) Covariates: sex, age + PCs from separate table ----------
dem_cases = cases[[ID_CASES, SEX_CASES, "age"]].rename(columns={ID_CASES:"IID", SEX_CASES:"sex"})
dem_ctrls = ctrls[[ID_CONTROLS, SEX_CONTROLS, "age"]].rename(columns={ID_CONTROLS:"IID", SEX_CONTROLS:"sex"})
for df in (dem_cases, dem_ctrls):
    df["IID"] = df["IID"].astype("string")

covar = pd.concat([dem_cases, dem_ctrls], ignore_index=True).drop_duplicates(subset=["IID"])
covar["FID"] = covar["IID"]
covar["sex"] = (covar["sex"].astype(str).str.strip().str.upper()
                .replace({"MALE":"M","FEMALE":"F","1":"M","2":"F","0":np.nan,"UNKNOWN":np.nan}))
covar["age"] = pd.to_numeric(covar["age"], errors="coerce")

# --- normalize & merge PCs ---
pcs_norm, pc_cols_found = normalize_pc_columns(PCs_final, PC_ID_COL, prefix=PC_PREFIX)
pcs_norm = pcs_norm.rename(columns={PC_ID_COL:"IID"})
pcs_norm["IID"] = pcs_norm["IID"].astype("string")

covar = covar.merge(pcs_norm, on="IID", how="left")

# order columns: FID IID sex age then PCs (numeric order)
def pc_key(name):
    m = re.search(r'(\d+)$', name)
    return (int(m.group(1)) if m else 10**9, name)
pc_order = sorted([c for c in covar.columns if c.upper().startswith(PC_PREFIX.upper())], key=pc_key)

covar = base.merge(covar[["FID","IID","sex","age"] + pc_order], on=["FID","IID"], how="left")
covar.to_csv("covar.tsv", sep="\t", index=False)

# ---------- 5) One-file version ----------
merged = covar.merge(pheno[["FID","IID","meningitis"]], on=["FID","IID"], how="left")
merged.to_csv("merged.tsv", sep="\t", index=False)

# ---------- 6) Variant list ----------
with open("variants.txt", "w") as fh:
    fh.write("rs17569141\n")

# ---------- 7) Diagnostics & previews ----------
try:
    print("Index date used for controls:", index_date.date())
except Exception:
    pass

print("PC columns detected & merged:", pc_cols_found if pc_cols_found else "None found")
# How many IDs overlapped between PCs and your base?
overlap_n = covar["IID"].notna().sum()
print(f"Samples in base: {len(base)}  | with any covariate row: {len(set(pd.concat([dem_cases['IID'], dem_ctrls['IID']])))}  | with PCs merged: {(~pcs_norm.drop(columns=['IID']).isna().all(axis=1)).sum()}")

print("\nWROTE: pheno.tsv, covar.tsv, merged.tsv, variants.txt")
print("N total:", len(base),
      "| cases:", int((pheno['meningitis']==1).sum()),
      "| controls:", int((pheno['meningitis']==0).sum()),
      "| missing pheno:", int(pheno['meningitis'].isna().sum()))

print("\npheno.tsv (head):")
display(pd.read_csv("pheno.tsv", sep="\t").head())

print("\ncovar.tsv (head):")
display(pd.read_csv("covar.tsv", sep="\t").head())

print("\nmerged.tsv (head):")
display(pd.read_csv("merged.tsv", sep="\t").head())


In [ ]:
import pandas as pd
import numpy as np
import re
from IPython.display import display # Needed for the .display() calls

def normalize_pc_columns(pcs_df, pc_id_col, prefix="PC"):
    """Rename variants of PC columns to 'PC<k>' (pc1, PC_01, 'PC 2' -> PC1/PC2), keep only ID + PCs."""
    df = pcs_df.copy()
    # strip spaces around names
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        if c == pc_id_col:
            continue
        s = str(c).strip()
        # match pc, PC, PC_, PC<space> and zero-padded numbers
        m = re.match(r'(?i)^pc[\s_]*0*([0-9]+)$', s)
        if not m:
            # also handle 'principal component 1' / 'pcscore1' styles if present
            m = re.match(r'(?i)^(principal[\s_]*component|pcscore)[\s_]*0*([0-9]+)$', s)
            if m:
                num = m.group(2)
                rename[c] = f'PC{int(num)}'
                continue
        if m:
            num = m.group(1)
            rename[c] = f'PC{int(num)}'
    df = df.rename(columns=rename)

    # keep ID + PC* columns only
    pc_cols = [c for c in df.columns if c != pc_id_col and str(c).upper().startswith(prefix.upper())]
    # ensure numeric PCs
    for c in pc_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # sort PC columns in numeric order (PC1, PC2, ..., PC10)
    def pc_num(name):
        m = re.search(r'(\d+)$', name)
        return int(m.group(1)) if m else 10**9

    pc_cols = sorted(pc_cols, key=pc_num)
    return df[[pc_id_col] + pc_cols], pc_cols


def create_analysis_files(
    # --- Data Inputs ---
    cases_df,
    controls_df,
    pcs_df,

    # --- Column Name Configs ---
    id_cases="person_id",
    id_controls="person_id",
    sex_cases="sex_at_birth",
    sex_controls="sex_at_birth",
    age_cases="age",
    age_controls="age",
    pc_id_col="research_id",
    pc_prefix="PC",

    # --- Output Configs ---
    phenotype_name="meningitis",
    pheno_outfile="pheno.tsv",
    covar_outfile="covar.tsv",
    merged_outfile="merged.tsv",
    variants_outfile="variants.txt",
    variant_list=["rs17569141"],

    # --- Other/Diagnostics ---
    index_date=None,
    write_files=True,
    show_previews=True
):
    """
    Processes case, control, and PC DataFrames to create phenotype,
    covariate, and merged files for analysis (e.g., GWAS).

    Args:
        cases_df (pd.DataFrame): DataFrame of cases.
        controls_df (pd.DataFrame): DataFrame of controls.
        pcs_df (pd.DataFrame): DataFrame with Principal Components.
        id_cases (str): ID column name in cases_df.
        id_controls (str): ID column name in controls_df.
        sex_cases (str): Sex column name in cases_df.
        sex_controls (str): Sex column name in controls_df.
        age_cases (str): Age column name in cases_df.
        age_controls (str): Age column name in controls_df.
        pc_id_col (str): ID column name in pcs_df.
        pc_prefix (str): Prefix for PC columns (e.g., "PC", "pc_").
        phenotype_name (str): Name for the phenotype column in output files.
        pheno_outfile (str): Output path for the phenotype file.
        covar_outfile (str): Output path for the covariate file.
        merged_outfile (str): Output path for the merged phenotype/covariate file.
        variants_outfile (str): Output path for the variant list file.
        variant_list (list): A list of variant IDs (e.g., rsIDs) to write.
        index_date (datetime.date, optional): Index date to print in diagnostics.
        write_files (bool): If True, write output files to disk.
        show_previews (bool): If True, print diagnostics and display file heads.

    Returns:
        tuple: (pheno_df, covar_df, merged_df)
    """

    # ---------- 1) Force ID dtypes to string everywhere ----------
    cases = cases_df.copy()
    ctrls = controls_df.copy()
    PCs_final = pcs_df.copy()

    cases[id_cases] = cases[id_cases].astype("string")
    ctrls[id_controls] = ctrls[id_controls].astype("string")
    PCs_final[pc_id_col] = PCs_final[pc_id_col].astype("string")

    # ---------- 2) Base sample list (union of IDs; FID=IID) ----------
    case_ids = pd.Series(cases[id_cases].dropna().unique(), dtype="string", name="IID")
    ctrl_ids = pd.Series(ctrls[id_controls].dropna().unique(), dtype="string", name="IID")
    base = pd.DataFrame({"IID": pd.concat([case_ids, ctrl_ids], ignore_index=True).drop_duplicates()})
    base["FID"] = base["IID"]
    base = base.astype({"FID":"string","IID":"string"})[["FID","IID"]]

    # ---------- 3) Phenotype (1=case, 0=control, NA otherwise) ----------
    pheno = base.copy()
    case_set, ctrl_set = set(case_ids), set(ctrl_ids)
    pheno[phenotype_name] = np.where(
        pheno["IID"].isin(case_set), 1,
        np.where(pheno["IID"].isin(ctrl_set), 0, np.nan)
    )
    if write_files:
        pheno.to_csv(pheno_outfile, sep="\t", index=False)

    # ---------- 4) Covariates: sex, age + PCs from separate table ----------
    dem_cases = cases[[id_cases, sex_cases, age_cases]].rename(columns={id_cases:"IID", sex_cases:"sex", age_cases:"age"})
    dem_ctrls = ctrls[[id_controls, sex_controls, age_controls]].rename(columns={id_controls:"IID", sex_controls:"sex", age_controls:"age"})
    
    for df in (dem_cases, dem_ctrls):
        df["IID"] = df["IID"].astype("string")

    covar = pd.concat([dem_cases, dem_ctrls], ignore_index=True).drop_duplicates(subset=["IID"])
    covar["FID"] = covar["IID"]
    covar["sex"] = (covar["sex"].astype(str).str.strip().str.upper()
                     .replace({"MALE":"M","FEMALE":"F","1":"M","2":"F","0":np.nan,"UNKNOWN":np.nan}))
    covar["age"] = pd.to_numeric(covar["age"], errors="coerce")

    # --- normalize & merge PCs ---
    pcs_norm, pc_cols_found = normalize_pc_columns(PCs_final, pc_id_col, prefix=pc_prefix)
    pcs_norm = pcs_norm.rename(columns={pc_id_col:"IID"})
    pcs_norm["IID"] = pcs_norm["IID"].astype("string")

    covar = covar.merge(pcs_norm, on="IID", how="left")

    # order columns: FID IID sex age then PCs (numeric order)
    def pc_key(name):
        m = re.search(r'(\d+)$', name)
        return (int(m.group(1)) if m else 10**9, name)
    
    pc_order = sorted([c for c in covar.columns if c.upper().startswith(pc_prefix.upper())], key=pc_key)

    covar = base.merge(covar[["FID","IID","sex","age"] + pc_order], on=["FID","IID"], how="left")
    
    if write_files:
        covar.to_csv(covar_outfile, sep="\t", index=False)

    # ---------- 5) One-file version ----------
    merged = covar.merge(pheno[["FID","IID", phenotype_name]], on=["FID","IID"], how="left")
    
    if write_files:
        merged.to_csv(merged_outfile, sep="\t", index=False)


    # ---------- 7) Diagnostics & previews ----------
    if show_previews:
        if index_date:
            try:
                print("Index date used for controls:", index_date.date())
            except Exception:
                print(f"Index date provided: {index_date}")

        print("PC columns detected & merged:", pc_cols_found if pc_cols_found else "None found")
        
        # Calculate overlaps
        base_covar_n = covar['sex'].notna().sum() + covar['age'].notna().sum()
        pc_overlap_n = covar[pc_order].notna().any(axis=1).sum()
        print(f"Samples in base: {len(base)}  | with any demo data (sex/age): {base_covar_n}  | with PCs merged: {pc_overlap_n}")

        print(f"\nWROTE: {', '.join(files_written)}")
        print("N total:", len(base),
              f"| cases:", int((pheno[phenotype_name]==1).sum()),
              f"| controls:", int((pheno[phenotype_name]==0).sum()),
              f"| missing pheno:", int(pheno[phenotype_name].isna().sum()))

        if write_files:
            print(f"\n{pheno_outfile} (head):")
            display(pd.read_csv(pheno_outfile, sep="\t").head())

            print(f"\n{covar_outfile} (head):")
            display(pd.read_csv(covar_outfile, sep="\t").head())

            print(f"\n{merged_outfile} (head):")
            display(pd.read_csv(merged_outfile, sep="\t").head())
            
    # ---------- 8) Return DataFrames ----------
    return pheno, covar, merged

In [ ]:
 # --- Load your data first (example) ---
# cases_df = pd.read_csv("my_cases.csv")
# controls_df = pd.read_csv("my_controls.csv")
# pcs_df = pd.read_csv("my_pcs.csv")
# some_index_date = pd.to_datetime("2023-01-01")

# --- Now, just call the function ---
pheno_data, covar_data, merged_data = create_analysis_files(
    cases_df= pd.read_csv("ns_vax_cases/covid_case_cohort.csv"),
    controls_df= pd.read_csv("ns_vax_controls/covid_controls.csv"),
    pcs_df=wrangled_pcs)
#     
#     # --- You only need to change parameters that are different ---
#     # --- from the defaults. For example: ---
#     
#     phenotype_name="disease_X",
#     pheno_outfile="disease_X.pheno.tsv",
#     covar_outfile="disease_X.covar.tsv",
#     merged_outfile="disease_X.merged.tsv",
#     variant_list=["rs123", "rs456"],
#     index_date=some_index_date
# )

# You can now work with the returned DataFrames
# print("\nReturned phenotype data info:")
# pheno_data.info()

In [ ]:
merged_data.head(15)

In [ ]:
merged.tail(10)

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display # Make sure this is imported

# ==================================================================
#  PASTE THE TWO FUNCTIONS FROM THE PREVIOUS ANSWER HERE
#
#  def normalize_pc_columns(...):
#      ...
#
#  def create_analysis_files(...):
#      ...
#
# ==================================================================


# --- 1. Setup Your Paths and Patterns ---

# Set the path to the folder containing your case/control files
DATA_FOLDER = Path("path/to/your/data_folder")

# Set the path to your single, shared PC file
PC_FILE_PATH = Path("path/to/your/PCs_final.csv") # Or .tsv, .parquet, etc.

# Define the naming convention for your files
CASE_SUFFIX = "_cases.csv"
CONTROL_SUFFIX = "_controls.csv"


# --- 2. Load the Single PC File (Assumption) ---
print(f"Loading shared PC file from: {PC_FILE_PATH}")
try:
    # Load your PC file once
    pcs_df_main = pd.read_csv(PC_FILE_PATH)
    # Or pd.read_parquet(PC_FILE_PATH), etc.
except FileNotFoundError:
    print(f"  [!] ERROR: PC file not found at {PC_FILE_PATH}.")
    print("  Please check the PC_FILE_PATH variable. Exiting.")
    # exit() # Uncomment to stop the script if PCs are required
except Exception as e:
    print(f"  [!] ERROR: Could not load PC file. {e}")
    # exit()


# --- 3. Find all "case" files to start the loop ---
# We'll find all case files, then find their matching control file.
case_files = list(DATA_FOLDER.glob(f"*{CASE_SUFFIX}"))

if not case_files:
    print(f"\nNo files found in {DATA_FOLDER} matching '*{CASE_SUFFIX}'")
else:
    print(f"\nFound {len(case_files)} case files. Starting processing...")

# --- 4. Loop and Run the Analysis ---
for case_file_path in case_files:
    
    # Extract the 'phenotype name' from the filename
    # e.g., "meningitis_cases.csv" -> "meningitis"
    pheno_name = case_file_path.name.replace(CASE_SUFFIX, "")
    
    print(f"\n" + "="*50)
    print(f"Processing phenotype: {pheno_name}")
    print(f"Loading case file: {case_file_path.name}")

    # Construct the expected control file name and path
    control_file_name = f"{pheno_name}{CONTROL_SUFFIX}"
    control_file_path = DATA_FOLDER / control_file_name

    # --- Safety Check: Ensure the matching control file exists ---
    if not control_file_path.exists():
        print(f"  [!] Skipping: Found {case_file_path.name}, but")
        print(f"  [!] Missing matching control file: {control_file_name}")
        continue
        
    print(f"Loading control file: {control_file_name}")

    try:
        # --- Load the data for this pair ---
        cases_df = pd.read_csv(case_file_path)
        controls_df = pd.read_csv(control_file_path)

        # --- Define unique output filenames for this phenotype ---
        # This is CRITICAL so you don't overwrite your files on each loop
        pheno_out = f"{pheno_name}_pheno.tsv"
        covar_out = f"{pheno_name}_covar.tsv"
        merged_out = f"{pheno_name}_merged.tsv"
        variants_out = f"{pheno_name}_variants.txt"

        print(f"  Running analysis for {pheno_name}...")
        
        # --- Call the main function ---
        pheno_data, covar_data, merged_data = create_analysis_files(
            cases_df=cases_df,
            controls_df=controls_df,
            pcs_df=pcs_df_main,  # Pass in the single PC file
            
            # --- Pass in the unique output filenames ---
            phenotype_name=pheno_name,
            pheno_outfile=pheno_out,
            covar_outfile=covar_out,
            merged_outfile=merged_out,
            variants_outfile=variants_out,
            
            # You can override other defaults here if needed
            # id_cases="person_id", 
            # pc_id_col="research_id",
        )
        
        print(f"  [+] Success: Finished processing for {pheno_name}.")

    except Exception as e:
        print(f"  [!] ERROR processing {pheno_name}: {e}")
        # This will catch errors (e.g., missing columns) in one
        # file pair but allow the loop to continue to the next pair.

print("\n" + "="*50)
print("All phenotype processing complete.")